In [ ]:
!pip install -q transformers datasets accelerate scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)


In [ ]:
#I already downloaded these two datasets from huggingface in a separate colab notebook


lmsys_path = "/content/drive/MyDrive/memctrl_datasets/lmsys-chat-1m.jsonl"
wildchat_path = "/content/drive/MyDrive/memctrl_datasets/wildchat.jsonl"

##### Analyzing the data to understand what it was made for so that I get an idea of how to process it for task classification purpose

In [ ]:
#Dataset 1: LMSYS Dataset

dataset1 = pd.read_json(lmsys_path, lines=True)
dataset1.head(10)

,conversation_id,model,conversation,turn,language,openai_moderation,redacted
0,f9ec8ce91221411c88de9f8a3d4e6a3d,dolly-v2-12b,[{'content': 'analyze the following patient's ...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
1,608f282d1d0248099c1f26e7a31a113c,vicuna-13b,[{'content': 'You are the text completion mode...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
2,7744651703d645afbc5c8fe1116b8293,vicuna-13b,[{'content': 'Is there a way to discover if th...,3,English,"[{'categories': {'harassment': False, 'harassm...",False
3,bac4e761977b4c31a0b5f2cbbf8d2c60,vicuna-13b,[{'content': 'Quieres independizarte y formar ...,7,Spanish,"[{'categories': {'harassment': False, 'harassm...",False
4,112b5025b8e34d8d9e2420a1d0d25a79,vicuna-13b,[{'content': 'You are an open-minded liberal d...,1,English,"[{'categories': {'harassment': False, 'harassm...",True
5,9e4bb9daac984bd7be03b98802add6f2,vicuna-13b,[{'content': 'Please tell me about a humanoid ...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
6,2435755fb39b4c41a31d251e10736e6e,alpaca-13b,[{'content': 'use alliteration with words lone...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
7,fa0ed971637a481e9a91fc8c21a96181,vicuna-13b,[{'content': 'Imagine you are a male college s...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
8,2d36928e73f847c5b3a26d77ccf48bd9,vicuna-13b,[{'content': 'There is an example: EBCM Autobr...,1,English,"[{'categories': {'harassment': False, 'harassm...",True
9,d3968d96081a4132bf947452247fdc49,koala-13b,[{'content': 'Give me an introduction over 200...,1,English,"[{'categories': {'harassment': False, 'harassm...",False


In [ ]:
#Dataset2: WildChat

dataset2 = pd.read_json(wildchat_path, lines=True)
dataset2.head(20)

,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted
0,26c5dc109920789f9199ff9b37acb8c1,gpt-4,2023-04-10 00:01:08+00:00,"[{'content': 'Write a very long, elaborate, de...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000102290490758, 'insul...",False,False
1,e87a1aeb9aafa35c00da39ddeb1139a0,gpt-4,2023-04-10 00:01:10+00:00,"[{'content': 'what are you?', 'language': 'Eng...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 9.160337503999473e-05, 'i...",False,False
2,c3415a9e401ff379f29fe3ce02e500dc,gpt-4,2023-04-10 00:02:37+00:00,[{'content': 'Write an engaging and a construc...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 9.502275497652592e-05, 'i...",False,False
3,ec6578cd9d69130a769ea307b6e7a874,gpt-4,2023-04-10 00:03:07+00:00,[{'content': 'CONSTRAINTS: 1. ~4000 word limi...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000127669045468, 'insul...",False,False
4,17827951de7dc8b29e8e2baa1a73e875,gpt-3.5-turbo,2023-04-10 00:03:09+00:00,[{'content': 'اكتب لي بحث عن اهميه نظام اتحاد ...,3,Arabic,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.004652907606214, 'insul...",False,False
5,c9fdedb0e222c9251e5fab2b0784240c,gpt-4,2023-04-10 00:03:58+00:00,[{'content': 'Write an engaging and a construc...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000102741440059, 'insul...",False,False
6,01e1f36d3f0cf0c4acb8e15bc7f4ed10,gpt-4,2023-04-10 00:04:48+00:00,[{'content': 'CONSTRAINTS: 1. ~4000 word limi...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000127669045468, 'insul...",False,False
7,bf21a7138ef8028bee411c1f271ef94c,gpt-4,2023-04-10 00:05:42+00:00,"[{'content': 'Write a very long, elaborate, de...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00044045384856800004, '...",False,False
8,a0d800583ab19bb20299af3925f9d6cf,gpt-4,2023-04-10 00:07:02+00:00,[{'content': 'CONSTRAINTS: 1. ~4000 word limi...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000127669045468, 'insul...",False,False
9,cc9c65c7200e499761e9b8bf6979f904,gpt-4,2023-04-10 00:07:14+00:00,"[{'content': 'Chris is quite innocent, shy, ch...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00015229957352800002, '...",False,False


In [ ]:
# Extracting contents inside the attributes

dataset2.iloc[9]["conversation"]

[{'content': "Chris is quite innocent, shy, cheerful, and has a heart of gold, though somewhat anti-social. He also seems to be quite soulful. In normal situations, Chris will try to be the mature one of the group providing information for his teammates when necessary.Chris appears as a well-mannered person, with him seemingly being a well-mannered young boy at a first glance. Despite his generally prim and proper decorum, it's been noted that Chris also possesses a murderous intent that is even stronger than Shermie's, which ties into him being able to 'kill with an innocent smile.' Alluding to this secretly violent behavior, Chris's dialogue can be a bit precocious and arrogant, and he tends to taunt opponents in a way to deliberately offend them, even his bandmates. Whether he is aware of his own haughtiness remains to be seen, Shermie is his girlfriend.\n\nMake a quote about Easter as Chris from king of fighters.",
  'language': 'English',
  'redacted': False,
  'role': 'user',
  '

In [ ]:
dataset2.iloc[9]["openai_moderation"]

[{'categories': {'harassment': False,
   'harassment/threatening': False,
   'hate': False,
   'hate/threatening': False,
   'self-harm': False,
   'self-harm/instructions': False,
   'self-harm/intent': False,
   'sexual': False,
   'sexual/minors': False,
   'violence': False,
   'violence/graphic': False},
  'category_scores': {'harassment': 0.0044180112890890005,
   'harassment/threatening': 0.000123215853818,
   'hate': 0.00017053210467500002,
   'hate/threatening': 1.58183843268489e-06,
   'self-harm': 4.29759566031862e-05,
   'self-harm/instructions': 7.871616617194377e-06,
   'self-harm/intent': 1.5881561921560202e-05,
   'sexual': 0.00021074917458400002,
   'sexual/minors': 4.695776078733615e-05,
   'violence': 0.11043757945299101,
   'violence/graphic': 1.982639105335693e-06},
  'flagged': False},
 {'categories': {'harassment': False,
   'harassment/threatening': False,
   'hate': False,
   'hate/threatening': False,
   'self-harm': False,
   'self-harm/instructions': False,


In [ ]:
dataset2.iloc[9]["detoxify_moderation"]

[{'identity_attack': 0.00015229957352800002,
  'insult': 0.010425202548503001,
  'obscene': 0.0004960042424490001,
  'severe_toxicity': 7.95356827438809e-05,
  'sexual_explicit': 0.00017336386372300002,
  'threat': 0.000145766767673,
  'toxicity': 0.013420132920145001},
 {'identity_attack': 7.477646431652829e-05,
  'insult': 0.00033976303529900003,
  'obscene': 0.00017197921988500002,
  'severe_toxicity': 1.833976239140611e-05,
  'sexual_explicit': 2.7393416530685503e-05,
  'threat': 2.864539237634744e-05,
  'toxicity': 0.000967103347647}]

In [ ]:
dataset2.iloc[16]["conversation"]

[{'content': 'Please print a chapter 1 of a book on teaching young children phonics and spelling ',
  'language': 'English',
  'redacted': False,
  'role': 'user',
  'toxic': False},
 {'content': "Chapter 1: Laying the Foundation for Phonics and Spelling Success\n\nIntroduction\n\nTeaching young children the essential skills of phonics and spelling is a critical component of early education. Through these foundational tools, children learn to understand the written language and become proficient readers and writers. This chapter will explore the importance of phonics and spelling in early literacy development and provide practical strategies for creating an engaging and effective learning environment.\n\nThe Importance of Phonics and Spelling\n\nPhonics is the relationship between letters and sounds in a language. It helps children understand the alphabetic code and learn to decode words. Young children must grasp this essential skill to develop reading and writing abilities, as phonic

In [ ]:
dataset2.iloc[16]["openai_moderation"]

[{'categories': {'harassment': False,
   'harassment/threatening': False,
   'hate': False,
   'hate/threatening': False,
   'self-harm': False,
   'self-harm/instructions': False,
   'self-harm/intent': False,
   'sexual': False,
   'sexual/minors': False,
   'violence': False,
   'violence/graphic': False},
  'category_scores': {'harassment': 1.268634059670148e-05,
   'harassment/threatening': 2.0874203983112242e-05,
   'hate': 8.128474291879684e-05,
   'hate/threatening': 4.0789920603856444e-05,
   'self-harm': 1.2577040742201e-08,
   'self-harm/instructions': 3.354158550905595e-08,
   'self-harm/intent': 1.96819627262812e-09,
   'sexual': 5.936019533692161e-06,
   'sexual/minors': 4.940969120070804e-06,
   'violence': 3.451535667409189e-05,
   'violence/graphic': 1.1333618203934743e-05},
  'flagged': False},
 {'categories': {'harassment': False,
   'harassment/threatening': False,
   'hate': False,
   'hate/threatening': False,
   'self-harm': False,
   'self-harm/instructions': Fa

In [ ]:
dataset2.iloc[16]["detoxify_moderation"]

[{'identity_attack': 0.000155811561853,
  'insult': 0.00042946074972800006,
  'obscene': 0.00015522299509,
  'severe_toxicity': 2.204470729338936e-05,
  'sexual_explicit': 2.8966245736228306e-05,
  'threat': 4.48017344751861e-05,
  'toxicity': 0.0016893188003450001},
 {'identity_attack': 0.000146647304063,
  'insult': 0.00023928291921000003,
  'obscene': 0.000350196147337,
  'severe_toxicity': 5.625836274703034e-05,
  'sexual_explicit': 5.008317384636029e-05,
  'threat': 5.762010187027044e-05,
  'toxicity': 0.000296603684546}]

In [ ]:
dataset1.head()

,conversation_id,model,conversation,turn,language,openai_moderation,redacted
0,f9ec8ce91221411c88de9f8a3d4e6a3d,dolly-v2-12b,[{'content': 'analyze the following patient's ...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
1,608f282d1d0248099c1f26e7a31a113c,vicuna-13b,[{'content': 'You are the text completion mode...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
2,7744651703d645afbc5c8fe1116b8293,vicuna-13b,[{'content': 'Is there a way to discover if th...,3,English,"[{'categories': {'harassment': False, 'harassm...",False
3,bac4e761977b4c31a0b5f2cbbf8d2c60,vicuna-13b,[{'content': 'Quieres independizarte y formar ...,7,Spanish,"[{'categories': {'harassment': False, 'harassm...",False
4,112b5025b8e34d8d9e2420a1d0d25a79,vicuna-13b,[{'content': 'You are an open-minded liberal d...,1,English,"[{'categories': {'harassment': False, 'harassm...",True


In [ ]:
total1 = len(dataset1)
total2 = len(dataset2)
print(f"Total rows in dataset1: {total1}")
print(f"Total rows in dataset2: {total2}")

Total rows in dataset1: 50000
Total rows in dataset2: 50000


In [ ]:
dataset2.head()

,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted
0,26c5dc109920789f9199ff9b37acb8c1,gpt-4,2023-04-10 00:01:08+00:00,"[{'content': 'Write a very long, elaborate, de...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000102290490758, 'insul...",False,False
1,e87a1aeb9aafa35c00da39ddeb1139a0,gpt-4,2023-04-10 00:01:10+00:00,"[{'content': 'what are you?', 'language': 'Eng...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 9.160337503999473e-05, 'i...",False,False
2,c3415a9e401ff379f29fe3ce02e500dc,gpt-4,2023-04-10 00:02:37+00:00,[{'content': 'Write an engaging and a construc...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 9.502275497652592e-05, 'i...",False,False
3,ec6578cd9d69130a769ea307b6e7a874,gpt-4,2023-04-10 00:03:07+00:00,[{'content': 'CONSTRAINTS: 1. ~4000 word limi...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.000127669045468, 'insul...",False,False
4,17827951de7dc8b29e8e2baa1a73e875,gpt-3.5-turbo,2023-04-10 00:03:09+00:00,[{'content': 'اكتب لي بحث عن اهميه نظام اتحاد ...,3,Arabic,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.004652907606214, 'insul...",False,False


##### The LMSYS and WildChat datasets look largely similar, with the main difference being their intent labels. From what I can tell, the WildChat dataset seems to have been designed more for intent- or sentiment-focused modeling (e.g., identifying toxicity, hate, or positive vs. negative conversational intent), likely for tasks such as troll detection.

##### Since my goal is task classification, I’ve decided to drop certain columns from both datasets so that their structures are exactly uniform. This way, I can cleanly merge them into a single, consolidated dataset.

In [ ]:
#dropping "openai_moderation" column from LMSYS dataset

lmsys = dataset1.drop(columns=["openai_moderation"])

In [ ]:
lmsys.head()

,conversation_id,model,conversation,turn,language,redacted
0,f9ec8ce91221411c88de9f8a3d4e6a3d,dolly-v2-12b,[{'content': 'analyze the following patient's ...,1,English,False
1,608f282d1d0248099c1f26e7a31a113c,vicuna-13b,[{'content': 'You are the text completion mode...,1,English,False
2,7744651703d645afbc5c8fe1116b8293,vicuna-13b,[{'content': 'Is there a way to discover if th...,3,English,False
3,bac4e761977b4c31a0b5f2cbbf8d2c60,vicuna-13b,[{'content': 'Quieres independizarte y formar ...,7,Spanish,False
4,112b5025b8e34d8d9e2420a1d0d25a79,vicuna-13b,[{'content': 'You are an open-minded liberal d...,1,English,True


In [ ]:
# dropping "openai_moderation", "detoxify_moderation", "timestamp", and "toxic" columns from Wildchat dataset

wildchat = dataset2.drop(columns=["openai_moderation", "detoxify_moderation", "timestamp", "toxic"])

In [ ]:
wildchat.head()

,conversation_id,model,conversation,turn,language,redacted
0,26c5dc109920789f9199ff9b37acb8c1,gpt-4,"[{'content': 'Write a very long, elaborate, de...",1,English,False
1,e87a1aeb9aafa35c00da39ddeb1139a0,gpt-4,"[{'content': 'what are you?', 'language': 'Eng...",1,English,False
2,c3415a9e401ff379f29fe3ce02e500dc,gpt-4,[{'content': 'Write an engaging and a construc...,1,English,False
3,ec6578cd9d69130a769ea307b6e7a874,gpt-4,[{'content': 'CONSTRAINTS: 1. ~4000 word limi...,1,English,False
4,17827951de7dc8b29e8e2baa1a73e875,gpt-3.5-turbo,[{'content': 'اكتب لي بحث عن اهميه نظام اتحاد ...,3,Arabic,False


In [ ]:
def extract_text(obj):
    if isinstance(obj, str):
        return obj
    if isinstance(obj, list):
        return " ".join(extract_text(x) for x in obj)
    if isinstance(obj, dict):
        return " ".join(extract_text(v) for v in obj.values())
    return ""

print(extract_text(wildchat.iloc[0]["conversation_id"]))

def load_jsonl(path):
    texts = []
    with open(path, "r") as f:
        for line in tqdm(f):
            obj = json.loads(line)
            text = extract_text(obj)
            if len(text.strip()) > 10:
                texts.append(text)
    return texts


26c5dc109920789f9199ff9b37acb8c1


In [ ]:
lmsys.head(0)

,conversation_id,model,conversation,turn,language,redacted


In [ ]:
wildchat.head(0)

,conversation_id,model,conversation,turn,language,redacted


In [ ]:
lmsys.info()
wildchat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   conversation_id  50000 non-null  object
 1   model            50000 non-null  object
 2   conversation     50000 non-null  object
 3   turn             50000 non-null  int64 
 4   language         50000 non-null  object
 5   redacted         50000 non-null  bool  
dtypes: bool(1), int64(1), object(4)
memory usage: 2.0+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   conversation_id  50000 non-null  object
 1   model            50000 non-null  object
 2   conversation     50000 non-null  object
 3   turn             50000 non-null  int64 
 4   language         50000 non-null  object
 5   redacted         50000 non-null  bool  
dtype

In [ ]:
raw_dataset = pd.concat([lmsys,wildchat], ignore_index=True)

In [ ]:
raw_dataset.head()

,conversation_id,model,conversation,turn,language,redacted
0,f9ec8ce91221411c88de9f8a3d4e6a3d,dolly-v2-12b,[{'content': 'analyze the following patient's ...,1,English,False
1,608f282d1d0248099c1f26e7a31a113c,vicuna-13b,[{'content': 'You are the text completion mode...,1,English,False
2,7744651703d645afbc5c8fe1116b8293,vicuna-13b,[{'content': 'Is there a way to discover if th...,3,English,False
3,bac4e761977b4c31a0b5f2cbbf8d2c60,vicuna-13b,[{'content': 'Quieres independizarte y formar ...,7,Spanish,False
4,112b5025b8e34d8d9e2420a1d0d25a79,vicuna-13b,[{'content': 'You are an open-minded liberal d...,1,English,True


In [ ]:
raw_dataset.duplicated(subset="conversation").sum()